# Plot spatial-cell tuning matrices

Read the single `.rfmap` saved by `spatial_cell_analysis.ipynb`. Display a selected unit and a 1D angular heatmap of all saved units. Run with the remote `~/.virtualenvs/rfmapping` kernel on `hhw9l84`.


In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from IPython.utils.capture import capture_output
from Utils.plotting import LIGHT_PLOT_STYLE
from Utils.rfmap import load_rf_maps, plot_2d_rfmap


In [ ]:
result_path = Path(
    "/mnt/senzailab/Kai/#Recording/m19/260831/260831_2"
    "/data/spatial_cells/ProbeA/baseline/egocentric_rate_map.rfmap"
)
rf_maps = load_rf_maps(result_path)
print(f"Available units: {rf_maps.unit_ids}")


In [ ]:
unit_id = 249
is_save = False
save_path = result_path.with_name(f"unit_{unit_id}.png")


rfmap = rf_maps.by_unit_id(unit_id)
# The RFMap helper shows immediately; defer display until the spatial labels are set.
with plt.rc_context({**LIGHT_PLOT_STYLE, "figure.figsize": (7, 5.5)}), capture_output():
    figure, axis = plot_2d_rfmap(rfmap.to_2d_array())

distance_edges = rfmap.metadata["xBinEdges"]
theta_edges = rfmap.metadata["yBinEdges"]
axis.images[0].set_extent((
    distance_edges[0], distance_edges[-1], theta_edges[-1], theta_edges[0],
))
axis.set_ylim(theta_edges[0], theta_edges[-1])
axis.set_aspect("auto")
axis.set_xticks(np.linspace(distance_edges[0], distance_edges[-1], 5))
axis.set_yticks(np.linspace(theta_edges[0], theta_edges[-1], 5))
axis.set_xlabel("Distance to boundary (cm)")
axis.set_ylabel("Egocentric angle (deg)")
axis.set_title(f"Egocentric tuning — unit {unit_id}")
figure.axes[1].set_ylabel("Hz")
figure.tight_layout()

if is_save:
    save_path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(save_path, dpi=300, facecolor="white", transparent=False)
display(figure)
plt.close(figure)


## All units: 1D angular heatmap

Sum each saved 2D matrix over distance, normalize each unit by its maximum, and sort by peak direction. The angle layout matches `hd_rf_comparison.ipynb`: 180 → 90 → 0 → 270 → 180. This displays the angular projection of the stored rates.


In [ ]:
is_save_heatmap = False
heatmap_path = result_path.with_name("all_units_angle_heatmap.png")

# Sum over distance, as in an RFMap y projection; unvisited bins stay missing.
rate_maps = rf_maps.to_2d_array()
angle_profiles = np.nansum(rate_maps, axis=2)
angle_profiles[~np.isfinite(rate_maps).any(axis=2)] = np.nan

# Use the same angular layout and peak ordering as hd_rf_comparison.ipynb.
angle_x = -((rf_maps[0].y_positions + 180.0) % 360.0 - 180.0)
column_order = np.argsort(angle_x)
profiles = angle_profiles[:, column_order]
row_max = np.max(np.where(np.isfinite(profiles), profiles, 0.0), axis=1, keepdims=True)
normalized_profiles = np.divide(
    profiles, row_max, out=np.zeros_like(profiles), where=row_max > 0,
)
normalized_profiles[~np.isfinite(profiles)] = np.nan
peak_bin = np.argmax(np.where(np.isfinite(profiles), profiles, -np.inf), axis=1)
# Keep silent or entirely missing units at the bottom, in their original order.
unit_order = np.argsort(
    np.where(row_max[:, 0] > 0, peak_bin, profiles.shape[1]), kind="stable",
)
sorted_unit_ids = np.asarray(rf_maps.unit_ids)[unit_order]
n_units = len(rf_maps)

# A stack of 1D curves is a 2D matrix; reuse the RFMap heatmap renderer.
with plt.rc_context({
    **LIGHT_PLOT_STYLE, "figure.figsize": (10, max(6, n_units * 0.16 + 1.5)),
}), capture_output():
    heatmap_figure, heatmap_axis = plot_2d_rfmap(normalized_profiles[unit_order])

heatmap_axis.images[0].set_extent((-180, 180, n_units - 0.5, -0.5))
heatmap_axis.images[0].set_clim(0, 1)
heatmap_axis.set_aspect("auto")
heatmap_axis.set_xticks([-180, -90, 0, 90, 180])
heatmap_axis.set_xticklabels(["180", "90", "0", "270", "180"])
heatmap_axis.set_yticks(np.arange(n_units))
heatmap_axis.set_yticklabels(sorted_unit_ids, fontsize=7)
heatmap_axis.set_xlabel("Egocentric angle (deg; RF spatial layout)")
heatmap_axis.set_ylabel("Unit ID")
heatmap_axis.set_title(f"Egocentric angular projection — all {n_units} units")
heatmap_figure.axes[1].set_ylabel("Normalized response")
heatmap_figure.tight_layout()

if is_save_heatmap:
    heatmap_path.parent.mkdir(parents=True, exist_ok=True)
    heatmap_figure.savefig(heatmap_path, dpi=300, facecolor="white", transparent=False)
display(heatmap_figure)
plt.close(heatmap_figure)
